In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:13pt;}
div.prompt {min-width:80px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:2px;}
table.dataframe{font-size:2xpt;}
div.text_cell_render h1 {
    color: #001A99 !important;
    font-weight: bold;
}
div.text_cell_render hr {
    border: 0 !important;
    height: 2px !important;
    background-color: #4FC3F7 !important;
}
</style>
"""))

#  벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장 (http://www.pinecone.io)에서 api key 받아 .env추가(PINECONE_API_KEY등록)
# 0. 패키지 설치

In [2]:
%pip install -q pinecone langchain-pinecone # --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


# 1. knowlege base 구성을 위한 데이터 생성

In [3]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 9.12 s
Wall time: 14.8 s


180

In [4]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')

In [ ]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embeding = UpstageEmbeddings(model = 'solar-embedding-1-large-passage')

In [ ]:
len(embedding.embed_query("샘플"))

In [ ]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [ ]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    # api_key=os.getenv('PINECONE_API_KEY')
)
# 데이터를 처음 저장할 때
index_name = 'tax-index'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드된 벡터 DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

# 2. 답변 생성을 위한 Retrieval

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [ ]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model = 'solar-pro2')

In [ ]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가이비다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""